# MNIST Classification and PCA


## 0. Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score, accuracy_score)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)
print("Imports ready.")

## Load MNIST and prepare the binary 7-versus-9 task

In [ ]:
# Download the 70,000 images (28x28 pixels flattened to 784 features).
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_all = mnist.data.astype(np.float32) / 255.0
y_all = mnist.target.astype(str)
print("Full dataset:", X_all.shape)

# Preserve the conventional first 60,000 training / last 10,000 test split.
Xtr_all, Xte_all = X_all[:60000], X_all[60000:]
ytr_all, yte_all = y_all[:60000], y_all[60000:]
tr = np.isin(ytr_all, ["7", "9"])
te = np.isin(yte_all, ["7", "9"])
X_train, y_train = Xtr_all[tr], (ytr_all[tr] == "9").astype(int)
X_test, y_test = Xte_all[te], (yte_all[te] == "9").astype(int)
print("Binary train:", X_train.shape, np.bincount(y_train))
print("Binary test:", X_test.shape, np.bincount(y_test))

In [ ]:
# Display five training images as a quick data check.
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(28, 28), cmap="gray")
    ax.set_title("Digit " + str(9 if y_train[i] else 7))
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 1 — Binary classifiers

### Exercise 1: Decision Tree using entropy

In [ ]:
tree = DecisionTreeClassifier(criterion="entropy", random_state=SEED)
tree.fit(X_train, y_train)
pred_tree = tree.predict(X_test)
print(classification_report(y_test, pred_tree, target_names=["7", "9"], zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):\n", confusion_matrix(y_test, pred_tree))
plt.figure(figsize=(18, 8))
plot_tree(tree, max_depth=3, class_names=["7", "9"], filled=True, rounded=True, fontsize=8)
plt.title("Decision tree (first three levels)")
plt.show()

### Exercise 2: Multinomial Naive Bayes

In [ ]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)
print(classification_report(y_test, pred_nb, target_names=["7", "9"], zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_nb))

### Exercise 3: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print(classification_report(y_test, pred_lr, target_names=["7", "9"], zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_lr))

### Exercise 4: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print(classification_report(y_test, pred_rf, target_names=["7", "9"], zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_rf))

## Part 2 — Evaluation metrics

In [ ]:
models = {"Decision Tree": tree, "Multinomial Naive Bayes": nb,
          "Logistic Regression": lr, "Random Forest": rf}

# Task 5a: classification report and confusion matrix for each model.
for name, model in models.items():
    print("\n", name)
    print(classification_report(y_test, model.predict(X_test),
                                target_names=["7", "9"], zero_division=0))
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test,
        display_labels=["7", "9"], cmap="Blues", values_format="d")
    plt.title(name + " confusion matrix")
    plt.tight_layout()
    plt.show()

In [ ]:
# Tasks 5b and 5c: ROC curves and ROC-AUC.
plt.figure(figsize=(8, 6))
for name, model in models.items():
    scores = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, scores)
    value = roc_auc_score(y_test, scores)
    plt.plot(fpr, tpr, label=f"{name} (AUC={value:.3f})")
    print(f"{name} ROC-AUC: {value:.4f}")
plt.plot([0, 1], [0, 1], "k--", label="Chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves (digit 9 is positive)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Task 5d: precision-recall curves. Average precision summarises each curve.
plt.figure(figsize=(8, 6))
for name, model in models.items():
    scores = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
    print(f"{name} average precision: {ap:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–recall curves (digit 9 is positive)")
plt.legend()
plt.tight_layout()
plt.show()

## Part 3 — Random Forest grid search and parallelisation

In [ ]:
# A reproducible training subset keeps the grid search manageable on a laptop.
rng = np.random.default_rng(SEED)
ids = rng.choice(len(y_train), size=min(10000, len(y_train)), replace=False)
X_tune, y_tune = X_train[ids], y_train[ids]
grid_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 20],
    "min_samples_leaf": [1, 2]
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
search = GridSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=1),
    param_grid=grid_params, scoring="accuracy", cv=cv, n_jobs=-1, verbose=1
)
start = time.perf_counter()
search.fit(X_tune, y_tune)
elapsed = time.perf_counter() - start
print("Best parameters:", search.best_params_)
print(f"Best cross-validation accuracy: {search.best_score_:.4f}")
print(f"Grid-search elapsed time: {elapsed:.1f} seconds")
print(f"Tuned model test accuracy: {accuracy_score(y_test, search.best_estimator_.predict(X_test)):.4f}")

## Part 4 — Multiclass classification and PCA

The following uses all ten digits. To keep runtime practical, it uses reproducible subsets from the standard MNIST train/test partitions. If the lecturer requires full-partition results, increase the sample sizes and document the runtime.

In [ ]:
X_multi_train_all, X_multi_test_all = X_all[:60000], X_all[60000:]
y_multi_train_all, y_multi_test_all = y_all[:60000], y_all[60000:]
rng = np.random.default_rng(SEED)
itr = rng.choice(len(y_multi_train_all), size=min(12000, len(y_multi_train_all)), replace=False)
ite = rng.choice(len(y_multi_test_all), size=min(2500, len(y_multi_test_all)), replace=False)
X_multi_train, y_multi_train = X_multi_train_all[itr], y_multi_train_all[itr]
X_multi_test, y_multi_test = X_multi_test_all[ite], y_multi_test_all[ite]
print("Multiclass train/test:", X_multi_train.shape, X_multi_test.shape)

In [ ]:
# Task 7a: One-vs-Rest LinearSVC without PCA.
baseline = OneVsRestClassifier(LinearSVC(random_state=SEED, dual="auto"))
start = time.perf_counter()
baseline.fit(X_multi_train, y_multi_train)
baseline_time = time.perf_counter() - start
baseline_pred = baseline.predict(X_multi_test)
baseline_acc = accuracy_score(y_multi_test, baseline_pred)
print(f"Training time without PCA: {baseline_time:.2f} seconds")
print(f"Test accuracy without PCA: {baseline_acc:.4f}")

In [ ]:
# Task 7b: fit PCA on training data only, then transform train and test.
pca = PCA(n_components=100, svd_solver="randomized", random_state=SEED)
start = time.perf_counter()
X_train_100 = pca.fit_transform(X_multi_train)
X_test_100 = pca.transform(X_multi_test)
pca_model = OneVsRestClassifier(LinearSVC(random_state=SEED, dual="auto"))
pca_model.fit(X_train_100, y_multi_train)
pca_time = time.perf_counter() - start
pca_pred = pca_model.predict(X_test_100)
pca_acc = accuracy_score(y_multi_test, pca_pred)
print(f"PCA transform + classifier training time: {pca_time:.2f} seconds")
print(f"Test accuracy with 100 components: {pca_acc:.4f}")

In [ ]:
# Task 7c: cumulative variance explained by the 100 retained components.
variance_100 = pca.explained_variance_ratio_.sum()
print(f"Cumulative variance explained by 100 components: {variance_100:.4%}")
cumvar = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, 101), cumvar)
plt.axhline(0.95, linestyle="--", label="95% target")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative variance explained")
plt.title("PCA cumulative explained variance")
plt.legend()
plt.tight_layout()
plt.show()
print("100 components reach 95%:", bool(variance_100 >= 0.95))